Single-Qubit Decoherence and Gate-Fidelity Simulation
======================================================

Cleaned standalone Python version of the original Colab script.

The script contains:
  - Stage 1-4 validation of unitary control and Lindblad decoherence;
  - gate-fidelity calculations and trajectory diagnostics;
  - Stage 6-8 parameter scans;
  - first-order weak-dissipation verification;
  - independent bivariate fits for the relaxation/dephasing coefficients;
  - solver-tolerance convergence checks;
  - second-order curvature formulas and numerical fits;
  - publication-oriented plotting functions.

Dependencies:
    numpy
    scipy
    matplotlib
    qutip >= 5

Units:
    The numerical examples use microseconds for time. Frequencies are
    therefore in rad / microsecond where applicable.

Scientific attribution:
    The first-order result
        1 - Fbar ~= (1/6) eps1 + (1/3) eps2
    is a specialization of Eq. (12) of Abad, Fernandez-Pendas, Kockum &
    Johansson, Phys. Rev. Lett. 129, 150504 (2022). The implementation here
    provides an independent numerical/integral cross-check; it should not be
    presented as a new analytic result.

    The general-theta second-order formulas are retained as the project's
    proposed extension, but the original script itself explicitly notes that
    an independent exhaustive literature check is still required before
    making a novelty claim.

In [ ]:
import os
import subprocess
import sys

In [ ]:
# ---------------------------------------------------------------------------
# Colab-friendly dependency setup
# ---------------------------------------------------------------------------
# QuTiP is not guaranteed to be preinstalled in a fresh Colab runtime.  The
# script therefore installs it automatically when necessary, so the entire
# project can be run with a single command/cell.
try:
    import qutip  # noqa: F401
except ImportError:
    print("QuTiP not found; installing it for this Colab/runtime ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "qutip"])

import matplotlib.pyplot as plt
import numpy as np
from scipy import integrate
from qutip import (
    Bloch,
    basis,
    destroy,
    expect,
    fidelity,
    mesolve,
    qeye,
    sigmax,
    sigmay,
    sigmaz,
)

In [ ]:
ZERO_H = 0 * qeye(2)

PROJ_1 = basis(2, 1) * basis(2, 1).dag()

CARDINAL_STATES = {'|0>': basis(2, 0), '|1>': basis(2, 1), '|+x>': (basis(2, 0) + basis(2, 1)).unit(), '|-x>': (basis(2, 0) - basis(2, 1)).unit(), '|+y>': (basis(2, 0) + 1j * basis(2, 1)).unit(), '|-y>': (basis(2, 0) - 1j * basis(2, 1)).unit()}

CARDINAL_BLOCH_VECTORS = [(0, 0, 1), (0, 0, -1), (1, 0, 0), (-1, 0, 0), (0, 1, 0), (0, -1, 0)]

In [ ]:
def Rx(theta):
    """Ideal rotation gate R_x(theta)."""
    return (-1j * theta / 2 * sigmax()).expm()

def Ry(theta):
    """NEW: Ideal rotation gate R_y(theta) = exp(-i theta sigma_y / 2)."""
    return (-1j * theta / 2 * sigmay()).expm()

def Rz(theta):
    """NEW: Ideal rotation gate R_z(theta) = exp(-i theta sigma_z / 2)."""
    return (-1j * theta / 2 * sigmaz()).expm()

GATES = {'x': (Rx, sigmax()), 'y': (Ry, sigmay()), 'z': (Rz, sigmaz())}

In [ ]:
def bloch_vector(state):
    return np.array([expect(sigmax(), state), expect(sigmay(), state), expect(sigmaz(), state)])

def lindblad_rates(T1, T2):
    if T1 <= 0 or T2 <= 0:
        raise ValueError('T1 and T2 must be positive.')
    inv_Tphi = 1.0 / T2 - 1.0 / (2.0 * T1)
    if inv_Tphi < -1e-12:
        raise ValueError(f'Unphysical input: T2={T2} > 2*T1={2 * T1}. Requires T2 <= 2*T1.')
    inv_Tphi = max(0.0, inv_Tphi)
    gamma_1 = 1.0 / T1
    gamma_phi = 0.5 * inv_Tphi
    return (gamma_1, gamma_phi)

def collapse_ops(T1, T2):
    gamma_1, gamma_phi = lindblad_rates(T1, T2)
    return [np.sqrt(gamma_1) * destroy(2), np.sqrt(gamma_phi) * sigmaz()]

In [ ]:
def final_average_fidelity(T1, T2, tg, theta_target=np.pi / 2, delta=0.0, axis='x', solver_options=None):
    """
    Average gate fidelity over the 6 cardinal states.

    NEW: `axis` in {"x","y","z"} selects the rotation axis of the drive and
    of the ideal comparison gate (default "x" reproduces v1 behaviour
    exactly). `solver_options` is passed through to mesolve (NEW: used by
    the solver-convergence check).
    """
    Rgate, sigma_axis = GATES[axis]
    Omega_actual = (theta_target + delta) / tg
    H = 0.5 * Omega_actual * sigma_axis
    c_ops = collapse_ops(T1, T2)
    U_ideal = Rgate(theta_target)
    kwargs = {}
    if solver_options is not None:
        kwargs['options'] = solver_options
    fids = []
    for psi0 in CARDINAL_STATES.values():
        res = mesolve(H, psi0, [0.0, tg], c_ops=c_ops, **kwargs)
        rho_final = res.states[-1]
        target = U_ideal * psi0
        fids.append(fidelity(rho_final, target) ** 2)
    return float(np.mean(fids))

In [ ]:
def _Tr_rho_D_rho(x, y, z, T1, T2):
    """
    Tr[rho D[rho]] for a pure state with Bloch vector (x,y,z), under the
    dissipator-only Bloch drift (dx/dt,dy/dt,dz/dt) = (-x/T2,-y/T2,-(z-1)/T1).
    Derived from d/dt Tr(rho^2) = x*dx/dt + y*dy/dt + z*dz/dt (dissipator part
    only), since Tr(rho^2) = (1+x^2+y^2+z^2)/2.
    """
    return -(x ** 2 + y ** 2) / (2 * T2) - z * (z - 1) / (2 * T1)

def _free_trajectory(bloch0, axis, phi):
    """Bloch vector after free (undamped) rotation by angle phi about `axis`."""
    x0, y0, z0 = bloch0
    if axis == 'x':
        return (x0, y0 * np.cos(phi) - z0 * np.sin(phi), y0 * np.sin(phi) + z0 * np.cos(phi))
    if axis == 'y':
        return (x0 * np.cos(phi) + z0 * np.sin(phi), y0, -x0 * np.sin(phi) + z0 * np.cos(phi))
    if axis == 'z':
        return (x0 * np.cos(phi) - y0 * np.sin(phi), x0 * np.sin(phi) + y0 * np.cos(phi), z0)
    raise ValueError("axis must be 'x', 'y', or 'z'")

In [ ]:
def analytic_ab_theory(theta_target=np.pi / 2, axis='x', T1=1.0, T2=1.0, tg=None):
    """
    Independent (from-scratch code, different method) reproduction of a
    KNOWN result: Eq. (12) of Abad et al., PRL 129, 150504 (2022),
    specialized to a single qubit. NOT a novel result -- see the module
    docstring's attribution note. Kept here because it is the calculation
    that surfaced NEW 2 (the marginal-scan bug) and because independent
    verification of a cited formula, with from-scratch code, is itself
    good practice worth reporting (in that spirit, not as a discovery).

    First-order interaction-picture (Dyson) perturbation theory in the
    dissipator gives, for each cardinal state j,

        1 - F_j  ~=  - Integral_0^tg  Tr[rho_free_j(s) D[rho_free_j(s)]] ds

    where rho_free_j(s) is the UNDAMPED trajectory of state j under the
    ideal rotation. This function evaluates that integral numerically
    (via scipy.integrate.quad, independent of QuTiP/mesolve) for the given
    axis and angle, then reports the implied (a, b) by evaluating the
    integral at two different (T1, T2) and solving the resulting linear
    system -- i.e. it does NOT hardcode 1/6, 1/3; those values should
    *emerge* from this calculation.

    Returns dict with 'coefficient_of_eps1', 'coefficient_of_eps2'
    (dimensionless), obtained by evaluating at unit tg and reading off the
    tg/T1, tg/T2 dependence directly (the integral is linear in 1/T1 and
    1/T2 separately, so two evaluations suffice).
    """
    if tg is None:
        tg = 1.0

    def total_infidelity(T1_, T2_):
        total = 0.0
        for bloch0 in CARDINAL_BLOCH_VECTORS:

            def integrand(s):
                phi = theta_target / tg * s
                x, y, z = _free_trajectory(bloch0, axis, phi)
                return _Tr_rho_D_rho(x, y, z, T1_, T2_)
            I_j, _ = integrate.quad(integrand, 0.0, tg)
            total += -I_j
        return total / 6.0
    eps1_a, eps2_a = (tg / T1, tg / T2)
    val_a = total_infidelity(T1, T2)
    T1b, T2b = (10 * T1, T2)
    val_b = total_infidelity(T1b, T2b)
    T1c, T2c = (T1, 10 * T2)
    val_c = total_infidelity(T1c, T2c)
    A = np.array([[eps1_a, eps2_a], [eps1_a / 10, eps2_a], [eps1_a, eps2_a / 10]])
    y = np.array([val_a, val_b, val_c])
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    return {'a_theory': coef[0], 'b_theory': coef[1]}

In [ ]:
def bivariate_ab_fit(eps1_grid, eps2_grid, tg=1.0, theta_target=np.pi / 2, axis='x'):
    """
    NEW: Independent least-squares fit of 1-Fbar = a*eps1 + b*eps2 + c over
    the full (unmasked) 2D (eps1, eps2) grid -- i.e. eps1 and eps2 are
    varied INDEPENDENTLY, not tied together via T2=2*T1 or T1->infinity as
    in the original marginal scans. This recovers the true, uncontaminated
    a and b, together with OLS standard errors and R^2.
    """
    rows, y = ([], [])
    for e1 in eps1_grid:
        for e2 in eps2_grid:
            if e2 < e1 / 2 - 1e-09:
                continue
            T1, T2 = (tg / e1, tg / e2)
            F = final_average_fidelity(T1, T2, tg, theta_target=theta_target, axis=axis)
            rows.append([e1, e2, 1.0])
            y.append(1.0 - F)
    X = np.array(rows)
    y = np.array(y)
    n, p = X.shape
    coef, residuals, rank, sv = np.linalg.lstsq(X, y, rcond=None)
    y_hat = X @ coef
    resid = y - y_hat
    dof = max(n - p, 1)
    sigma2 = np.sum(resid ** 2) / dof
    XtX_inv = np.linalg.inv(X.T @ X)
    se = np.sqrt(np.diag(sigma2 * XtX_inv))
    ss_res = np.sum(resid ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return {'a': coef[0], 'a_se': se[0], 'b': coef[1], 'b_se': se[1], 'intercept': coef[2], 'intercept_se': se[2], 'r2': r2, 'n_points': n}

In [ ]:
def second_order_theory(theta_target, axis='x'):
    """
    NEW (pending the authors' own literature check -- see module
    docstring): closed-form second-order weak-dissipation coefficients for
    ARBITRARY theta, generalizing Abad et al.'s theta=pi-only Eqs. (13)-(14).

    Returns (c11, c22, c12) such that
        1 - Fbar ~= a*eps1 + b*eps2 - (c11*eps1**2 + c22*eps2**2 + c12*eps1*eps2)
    with a=1/6, b=1/3 (NEW 1 / Abad Eq. 12).

    axis="y" is identical to axis="x" by the noise model's azimuthal
    symmetry about z (verified numerically); axis="z" is theta-independent.
    """
    if axis in ('x', 'y'):
        th = theta_target
        c11 = (th ** 2 + np.sin(th) ** 2) / (24 * th ** 2)
        c22 = 1.0 / 8 + np.sin(th) ** 2 / (24 * th ** 2)
        c12 = (th ** 2 - np.sin(th) ** 2) / (12 * th ** 2)
        return (c11, c22, c12)
    elif axis == 'z':
        return (1.0 / 12, 1.0 / 6, 0.0)
    else:
        raise ValueError("axis must be 'x', 'y', or 'z'")

In [ ]:
def weak_dissipation_prediction(eps1, eps2, theta_target=np.pi / 2, axis='x'):
    """Full first+second-order analytic prediction of 1 - Fbar (NEW 1 + NEW 6)."""
    a_true, b_true = (1.0 / 6, 1.0 / 3)
    c11, c22, c12 = second_order_theory(theta_target, axis=axis)
    linear = a_true * eps1 + b_true * eps2
    quadratic = c11 * eps1 ** 2 + c22 * eps2 ** 2 + c12 * eps1 * eps2
    return linear - quadratic

In [ ]:
def quadratic_ab_fit(eps1_grid, eps2_grid, tg=1.0, theta_target=np.pi / 2, axis='x'):
    """
    NEW: Extends the linear model to
        1-Fbar = a*eps1 + b*eps2 + c11*eps1^2 + c22*eps2^2 + c12*eps1*eps2
    to quantify curvature and test whether it accounts for the ~20%
    linear-fit under-prediction noted for the worked example
    (T1=50us, T2=30us, tg=40ns => eps1=8e-4, eps2=1.33e-3) in the draft.
    """
    rows, y = ([], [])
    for e1 in eps1_grid:
        for e2 in eps2_grid:
            if e2 < e1 / 2 - 1e-09:
                continue
            T1, T2 = (tg / e1, tg / e2)
            F = final_average_fidelity(T1, T2, tg, theta_target=theta_target, axis=axis)
            rows.append([e1, e2, e1 ** 2, e2 ** 2, e1 * e2])
            y.append(1.0 - F)
    X = np.array(rows)
    y = np.array(y)
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    y_hat = X @ coef
    ss_res = np.sum((y - y_hat) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return {'a': coef[0], 'b': coef[1], 'c11': coef[2], 'c22': coef[3], 'c12': coef[4], 'r2': r2}

In [ ]:
def solver_convergence_check(T1=50.0, T2=30.0, tg=0.04, theta_target=np.pi / 2, tolerances=(1e-06, 1e-08, 1e-10, 1e-12)):
    """
    NEW: Confirms the reported fidelity is stable under mesolve's default
    ODE tolerances, i.e. that the precision claimed in the validation
    table (Sec. 4.1 of the draft) reflects physics/solver defaults rather
    than being an artifact that shifts under tighter tolerances.
    """
    results = {}
    for tol in tolerances:
        opts = {'atol': tol, 'rtol': tol}
        F = final_average_fidelity(T1, T2, tg, theta_target=theta_target, solver_options=opts)
        results[tol] = F
    F_vals = np.array(list(results.values()))
    spread = F_vals.max() - F_vals.min()
    return (results, spread)

In [ ]:
def make_new_plots(outdir='figures_v2'):
    import os
    os.makedirs(outdir, exist_ok=True)
    eps_maxes = np.array([0.03, 0.02, 0.01, 0.005, 0.002, 0.001, 0.0005])
    a_errs, b_errs = ([], [])
    for eps_max in eps_maxes:
        eps_grid = np.linspace(eps_max / 10, eps_max, 6)
        fit = bivariate_ab_fit(eps_grid, eps_grid, theta_target=np.pi / 2)
        a_errs.append(abs(fit['a'] - 1 / 6))
        b_errs.append(abs(fit['b'] - 1 / 3))
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.loglog(eps_maxes, a_errs, 'o-', label='$|a_{\\rm fit}-1/6|$')
    ax.loglog(eps_maxes, b_errs, 's-', label='$|b_{\\rm fit}-1/3|$')
    ax.loglog(eps_maxes, 0.1 * eps_maxes, 'k--', alpha=0.6, label='$\\propto \\epsilon_{\\max}$ (reference)')
    ax.set_xlabel('Fit-window upper bound $\\epsilon_{\\max}$')
    ax.set_ylabel('Absolute error vs. theory')
    ax.set_title('Convergence of the fitted bivariate coefficients\nto the analytic weak-dissipation values')
    ax.legend()
    ax.grid(True, which='both', ls='--', alpha=0.5)
    fig.tight_layout()
    fig.savefig(f'{outdir}/plot12_ab_convergence.png', dpi=150)
    plt.close(fig)
    axes_list = ['x', 'y', 'z']
    a_true_theory = 1 / 6
    b_true_theory = 1 / 3
    predicted_contaminated = a_true_theory + b_true_theory / 2
    eps_list = np.linspace(0.0005, 0.005, 8)
    marginal_a = []
    true_a = []
    for axis in axes_list:
        F1 = stage7_marginal_scan(eps_list, vary='eps1', axis=axis)
        marginal_a.append(np.polyfit(eps_list, 1 - F1, 1)[0])
        fit = bivariate_ab_fit(eps_list, eps_list, theta_target=np.pi / 2, axis=axis)
        true_a.append(fit['a'])
    fig, ax = plt.subplots(figsize=(6, 4.5))
    x_pos = np.arange(len(axes_list))
    width = 0.3
    ax.bar(x_pos - width, marginal_a, width, label="marginal-scan '$a$' ($T_2{=}2T_1$ forced)")
    ax.bar(x_pos, true_a, width, label='true bivariate $a$ (independent fit)')
    ax.axhline(predicted_contaminated, color='C0', ls='--', lw=1, label='predicted: $a_{\\rm true}+b_{\\rm true}/2=1/3$')
    ax.axhline(a_true_theory, color='C1', ls='--', lw=1, label='$a_{\\rm true}=1/6$')
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'R{a}' for a in axes_list])
    ax.set_ylabel('Fitted coefficient')
    ax.set_title("Marginal-scan 'a' is contaminated by the T2<=2T1 constraint;\nthe true bivariate a is a factor of 2 smaller")
    ax.legend(fontsize=8)
    ax.grid(True, axis='y', ls='--', alpha=0.5)
    fig.tight_layout()
    fig.savefig(f'{outdir}/plot13_marginal_vs_true_a.png', dpi=150)
    plt.close(fig)
    print(f'Plots 12-13 written to ./{outdir}/')

In [ ]:
def make_second_order_plot(outdir='figures_v2'):
    """NEW: Plot 14 -- theta-dependence of the closed-form curvature coefficients."""
    import os
    os.makedirs(outdir, exist_ok=True)
    thetas = np.linspace(0.05, 2 * np.pi, 400)
    c11x = np.array([second_order_theory(th, axis='x')[0] for th in thetas])
    c22x = np.array([second_order_theory(th, axis='x')[1] for th in thetas])
    c12x = np.array([second_order_theory(th, axis='x')[2] for th in thetas])
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(thetas, c11x, label='$c_{11}(\\theta)$, axis $x$')
    ax.plot(thetas, c22x, label='$c_{22}(\\theta)$, axis $x$')
    ax.plot(thetas, c12x, label='$c_{12}(\\theta)$, axis $x$')
    ax.axhline(1 / 12, ls='--', color='gray', lw=1, label='axis $z$: $c_{11}=1/12$ (const.)')
    ax.axhline(1 / 6, ls=':', color='gray', lw=1, label='axis $z$: $c_{22}=1/6$ (const.)')
    tested = [np.pi / 4, np.pi / 2, np.pi, 3 * np.pi / 2]
    labels = ['$\\pi/4$', '$\\pi/2$', '$\\pi$', '$3\\pi/2$']
    for th, lab in zip(tested, labels):
        ax.axvline(th, color='k', alpha=0.12, lw=1)
    ax.plot([np.pi], [second_order_theory(np.pi, 'x')[0]], 'ko', ms=6, zorder=5)
    ax.annotate('Abad et al.\n(theta=pi, published)', xy=(np.pi, second_order_theory(np.pi, 'x')[0]), xytext=(np.pi + 0.3, 0.02), fontsize=8)
    ax.set_xlabel('Target rotation angle $\\theta$')
    ax.set_ylabel('Curvature coefficient')
    ax.set_title('Second-order (curvature) coefficients vs. target angle\n(closed form; z-axis is theta-independent, x/y is not)')
    ax.set_xticks(tested)
    ax.set_xticklabels(labels)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, ls='--', alpha=0.4)
    fig.tight_layout()
    fig.savefig(f'{outdir}/plot14_second_order_theta_dependence.png', dpi=150)
    plt.close(fig)
    print(f'Plot 14 written to ./{outdir}/')

In [ ]:
def gate_trajectory(T1, T2, tg, theta_target=np.pi / 2, delta=0.0, num_pts=150):
    """
    Fidelity-vs-time *within* a single gate execution (Sec. 9 Stage 5).
    The reference target at each intermediate time t is the ideal,
    error-free, decoherence-free instantaneous rotation R_x(theta_target * t/tg),
    i.e. "are we tracking the intended path as the pulse proceeds".

    Integrates once over the full tlist per cardinal state (FIX 4), instead
    of restarting from t=0 for every point.
    """
    Omega_ideal = theta_target / tg
    Omega_actual = (theta_target + delta) / tg
    H = 0.5 * Omega_actual * sigmax()
    c_ops = collapse_ops(T1, T2)
    tlist = np.linspace(0.0, tg, num_pts)
    traj_fids = []
    for psi0 in CARDINAL_STATES.values():
        res = mesolve(H, psi0, tlist, c_ops=c_ops)
        f_t = np.array([fidelity(rho_t, Rx(Omega_ideal * t) * psi0) ** 2 for rho_t, t in zip(res.states, tlist)])
        traj_fids.append(f_t)
    avg_fidelity = np.mean(traj_fids, axis=0)
    return (tlist, avg_fidelity)

In [ ]:
def stage1_unitary_validation(theta_list=None):
    """Verify P1(theta) = sin^2(theta/2) for ideal R_x(theta)|0>."""
    if theta_list is None:
        theta_list = np.linspace(0, 2 * np.pi, 200)
    P1_numeric = np.array([expect(PROJ_1, Rx(th) * basis(2, 0)) for th in theta_list])
    P1_analytic = np.sin(theta_list / 2) ** 2
    return (theta_list, P1_numeric, P1_analytic)

def stage2_T1_validation(T1, tlist):
    """Verify P1(t) = exp(-t/T1) for free decay of |1>."""
    c_ops = [np.sqrt(1.0 / T1) * destroy(2)]
    res = mesolve(ZERO_H, basis(2, 1), tlist, c_ops=c_ops, e_ops=[PROJ_1])
    P1_numeric = res.expect[0]
    P1_analytic = np.exp(-tlist / T1)
    return (P1_numeric, P1_analytic)

def stage3_dephasing_validation(Tphi, tlist):
    """Verify x(t) = exp(-t/T_phi) for |+>, populations unchanged (z(t)=0)."""
    gamma_phi = 1.0 / (2.0 * Tphi)
    c_ops = [np.sqrt(gamma_phi) * sigmaz()]
    psi0 = (basis(2, 0) + basis(2, 1)).unit()
    res = mesolve(ZERO_H, psi0, tlist, c_ops=c_ops, e_ops=[sigmax(), sigmaz()])
    x_numeric, z_numeric = res.expect
    x_analytic = np.exp(-tlist / Tphi)
    return (x_numeric, z_numeric, x_analytic)

def analytic_bloch_trajectory(t, T1, T2, omega0, r0):
    """Exact solution of the Bloch-Slichter equations, Sec. 7."""
    x0, y0, z0 = r0
    x = np.exp(-t / T2) * (x0 * np.cos(omega0 * t) - y0 * np.sin(omega0 * t))
    y = np.exp(-t / T2) * (y0 * np.cos(omega0 * t) + x0 * np.sin(omega0 * t))
    z_eq = 1.0
    z = z_eq + (z0 - z_eq) * np.exp(-t / T1)
    return (x, y, z)

def stage4_combined_benchmark(T1, T2, omega0, tlist, psi0=None):
    """Numeric Lindblad evolution vs. exact analytic trajectory, free precession."""
    if psi0 is None:
        psi0 = (basis(2, 0) + basis(2, 1)).unit()
    H = 0.5 * omega0 * sigmaz()
    c_ops = collapse_ops(T1, T2)
    res = mesolve(H, psi0, tlist, c_ops=c_ops, e_ops=[sigmax(), sigmay(), sigmaz()])
    x_num, y_num, z_num = res.expect
    r0 = bloch_vector(psi0)
    x_an, y_an, z_an = analytic_bloch_trajectory(tlist, T1, T2, omega0, r0)
    purity = 0.5 * (1 + x_num ** 2 + y_num ** 2 + z_num ** 2)
    return {'numeric': (x_num, y_num, z_num), 'analytic': (x_an, y_an, z_an), 'purity': purity}

In [ ]:
def stage6_fidelity_vs_gate_time(T1, T2, tg_list, theta_target=np.pi / 2):
    return np.array([final_average_fidelity(T1, T2, tg, theta_target) for tg in tg_list])

In [ ]:
def stage7_marginal_scan(eps_list, vary='eps1', tg=1.0, theta_target=np.pi / 2, T_off=1000000.0, axis='x'):
    F = np.zeros_like(eps_list, dtype=float)
    for i, eps in enumerate(eps_list):
        T = tg / eps
        if vary == 'eps1':
            F[i] = final_average_fidelity(T1=T, T2=2 * T, tg=tg, theta_target=theta_target, axis=axis)
        elif vary == 'eps2':
            F[i] = final_average_fidelity(T1=T_off, T2=T, tg=tg, theta_target=theta_target, axis=axis)
        else:
            raise ValueError("vary must be 'eps1' or 'eps2'")
    return F

def stage7_2d_scan(eps1_list, eps2_list, tg=1.0, theta_target=np.pi / 2, axis='x'):
    F_grid = np.full((len(eps2_list), len(eps1_list)), np.nan)
    for i, e1 in enumerate(eps1_list):
        for j, e2 in enumerate(eps2_list):
            if e2 < e1 / 2 - 1e-09:
                continue
            T1, T2 = (tg / e1, tg / e2)
            F_grid[j, i] = final_average_fidelity(T1, T2, tg, theta_target, axis=axis)
    return F_grid

In [ ]:
def stage8_control_error_scan(delta_list, eps_list, tg=1.0, theta_target=np.pi / 2):
    """
    F(delta, eps) with T1 = T2 = T (eps1 = eps2 = eps), i.e. a single
    combined decoherence parameter, so the 3-parameter surface reduces to
    a plottable 2D heatmap.
    """
    F_grid = np.zeros((len(eps_list), len(delta_list)))
    for i, d in enumerate(delta_list):
        for j, eps in enumerate(eps_list):
            T = tg / eps
            F_grid[j, i] = final_average_fidelity(T, T, tg, theta_target, delta=d)
    return F_grid

In [ ]:
def stage10_axis_independence_scan(theta_targets=None, axes=('x', 'y', 'z'), eps_max=0.01, n_per_axis=6, tg=1.0):
    """
    Numerically confirms gate-AXIS independence (Rx vs Ry vs Rz) at first
    order. NOTE: this is already guaranteed by the fully general statement
    of Abad et al.'s theorem ("independent of the specific operation") --
    it is NOT a stronger test than their published claim, just one more
    numerical instance of it. Useful as a from-scratch code sanity check
    (and as the baseline for where axis-dependence *does* appear -- see
    NEW 6, second-order curvature, which is NOT axis-independent). Uses
    the proper bivariate fit (NEW 2), not the contaminated marginal scan.
    """
    if theta_targets is None:
        theta_targets = [np.pi / 4, np.pi / 2, np.pi, 3 * np.pi / 2]
    eps_grid = np.linspace(eps_max / 10, eps_max, n_per_axis)
    results = {}
    for axis in axes:
        for theta in theta_targets:
            fit = bivariate_ab_fit(eps_grid, eps_grid, tg=tg, theta_target=theta, axis=axis)
            results[axis, theta] = fit
    return results

In [ ]:
def make_all_plots(outdir='figures'):
    import os
    os.makedirs(outdir, exist_ok=True)
    thetas = np.linspace(0, np.pi, 60)
    pts = np.array([bloch_vector(Rx(th) * basis(2, 0)) for th in thetas]).T
    b = Bloch()
    b.add_points(pts, meth='l')
    b.add_points(pts[:, [0, -1]], meth='s')
    b.render()
    b.fig.savefig(f'{outdir}/plot1_ideal_bloch_trajectory.png', dpi=150)
    plt.close(b.fig)
    T1 = 50.0
    tlist = np.linspace(0, 4 * T1, 200)
    P1_num, P1_an = stage2_T1_validation(T1, tlist)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(tlist, P1_num, 'b-', lw=2, label='Lindblad (numeric)')
    ax.plot(tlist, P1_an, 'k--', lw=1.5, label='analytic $e^{-t/T_1}$')
    ax.set_xlabel('Time $t\\ (\\mu s)$')
    ax.set_ylabel('$P_1(t)$')
    ax.set_title(f'Stage 2: $T_1$ Relaxation Validation ($T_1={T1}\\mu s$)')
    ax.legend()
    ax.grid(True, ls='--', alpha=0.6)
    fig.tight_layout()
    fig.savefig(f'{outdir}/plot2_T1_decay.png', dpi=150)
    plt.close(fig)
    Tphi = 40.0
    tlist = np.linspace(0, 4 * Tphi, 200)
    x_num, z_num, x_an = stage3_dephasing_validation(Tphi, tlist)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(tlist, x_num, 'r-', lw=2, label='Lindblad (numeric) $x(t)$')
    ax.plot(tlist, x_an, 'k--', lw=1.5, label='analytic $e^{-t/T_\\phi}$')
    ax.plot(tlist, z_num, 'g-', lw=1.5, label='$z(t)$ (should stay 0)')
    ax.set_xlabel('Time $t\\ (\\mu s)$')
    ax.set_ylabel('Bloch component')
    ax.set_title(f'Stage 3: Pure Dephasing Validation ($T_\\phi={Tphi}\\mu s$)')
    ax.legend()
    ax.grid(True, ls='--', alpha=0.6)
    fig.tight_layout()
    fig.savefig(f'{outdir}/plot3_dephasing_decay.png', dpi=150)
    plt.close(fig)
    T1c, T2c, omega0 = (50.0, 30.0, 2 * np.pi * 0.05)
    tlist = np.linspace(0, 60, 300)
    out4 = stage4_combined_benchmark(T1c, T2c, omega0, tlist)
    (x_num, y_num, z_num), (x_an, y_an, z_an) = (out4['numeric'], out4['analytic'])
    fig, axes = plt.subplots(3, 1, figsize=(7, 8), sharex=True)
    for ax, num, an, label in zip(axes, (x_num, y_num, z_num), (x_an, y_an, z_an), 'xyz'):
        ax.plot(tlist, num, 'b-', lw=2, label='numeric')
        ax.plot(tlist, an, 'k--', lw=1.5, label='analytic')
        ax.set_ylabel(f'${label}(t)$')
        ax.legend(loc='upper right')
        ax.grid(True, ls='--', alpha=0.6)
    axes[-1].set_xlabel('Time $t\\ (\\mu s)$')
    fig.suptitle(f'Stage 4: Combined Dissipation Benchmark ($T_1={T1c}, T_2={T2c}\\mu s$)')
    fig.tight_layout()
    fig.savefig(f'{outdir}/plot4_combined_trajectory.png', dpi=150)
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(tlist, out4['purity'], 'm-', lw=2)
    ax.axhline(0.5, color='gray', ls=':', label='maximally mixed')
    ax.set_xlabel('Time $t\\ (\\mu s)$')
    ax.set_ylabel('Purity $\\mathcal{P}(t)$')
    ax.set_title('Stage 4: Purity Evolution')
    ax.legend()
    ax.grid(True, ls='--', alpha=0.6)
    fig.tight_layout()
    fig.savefig(f'{outdir}/plot5_purity.png', dpi=150)
    plt.close(fig)
    T1d, T2d = (50.0, 30.0)
    tg_list = np.logspace(-3, -0.5, 40)
    F_vs_tg = stage6_fidelity_vs_gate_time(T1d, T2d, tg_list)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.loglog(tg_list, 1 - F_vs_tg, 'b.-')
    ax.set_xlabel('Gate duration $t_g\\ (\\mu s)$')
    ax.set_ylabel('Infidelity $1-\\bar F$')
    ax.set_title(f'Stage 6: Gate Infidelity vs. Duration ($T_1={T1d}, T_2={T2d}\\mu s$)')
    ax.grid(True, which='both', ls='--', alpha=0.6)
    fig.tight_layout()
    fig.savefig(f'{outdir}/plot6_fidelity_vs_tg.png', dpi=150)
    plt.close(fig)
    eps_list = np.logspace(-3, np.log10(0.4), 30)
    F_eps1 = stage7_marginal_scan(eps_list, vary='eps1')
    F_eps2 = stage7_marginal_scan(eps_list, vary='eps2')
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(eps_list, 1 - F_eps1, 'b.-')
    ax.set_xlabel('$\\epsilon_1 = t_g/T_1$')
    ax.set_ylabel('Infidelity $1-\\bar F$')
    ax.set_title('Stage 7: Infidelity vs. $\\epsilon_1$ ($T_2 \\to \\infty$)')
    ax.grid(True, ls='--', alpha=0.6)
    fig.tight_layout()
    fig.savefig(f'{outdir}/plot7_fidelity_vs_eps1.png', dpi=150)
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(eps_list, 1 - F_eps2, 'r.-')
    ax.set_xlabel('$\\epsilon_2 = t_g/T_2$')
    ax.set_ylabel('Infidelity $1-\\bar F$')
    ax.set_title('Stage 7: Infidelity vs. $\\epsilon_2$ ($T_1 \\to \\infty$)')
    ax.grid(True, ls='--', alpha=0.6)
    fig.tight_layout()
    fig.savefig(f'{outdir}/plot8_fidelity_vs_eps2.png', dpi=150)
    plt.close(fig)
    eps1_grid = np.linspace(0.005, 0.35, 25)
    eps2_grid = np.linspace(0.005, 0.35, 25)
    F_grid = stage7_2d_scan(eps1_grid, eps2_grid)
    fig, ax = plt.subplots(figsize=(6, 5))
    cf = ax.contourf(eps1_grid, eps2_grid, F_grid, levels=20, cmap='viridis')
    fig.colorbar(cf, ax=ax, label='$\\bar F(\\epsilon_1,\\epsilon_2)$')
    ax.set_xlabel('$\\epsilon_1 = t_g/T_1$')
    ax.set_ylabel('$\\epsilon_2 = t_g/T_2$')
    ax.set_title('Stage 7: 2D Fidelity Map (masked where $T_2>2T_1$)')
    fig.tight_layout()
    fig.savefig(f'{outdir}/plot9_fidelity_2d_map.png', dpi=150)
    plt.close(fig)
    delta_grid = np.linspace(-0.3, 0.3, 25)
    eps_grid = np.linspace(0.005, 0.3, 25)
    F_ctrl = stage8_control_error_scan(delta_grid, eps_grid)
    fig, ax = plt.subplots(figsize=(6, 5))
    cf = ax.contourf(delta_grid, eps_grid, 1 - F_ctrl, levels=20, cmap='magma')
    fig.colorbar(cf, ax=ax, label='Infidelity $1-\\bar F$')
    ax.set_xlabel('Control error $\\delta$ (rad)')
    ax.set_ylabel('$\\epsilon = t_g/T_1 = t_g/T_2$')
    ax.set_title('Stage 8: Composite Infidelity Map')
    fig.tight_layout()
    fig.savefig(f'{outdir}/plot10_control_error_map.png', dpi=150)
    plt.close(fig)
    print(f'All 10 plots written to ./{outdir}/')

In [ ]:
def main(generate_plots=False, generate_extended_plots=False):
    """Run numerical validation and, optionally, generate all figures."""

    print("=" * 72)
    print("Stage 1-4: basic numerical validation")
    print("=" * 72)

    _, p1_numeric, p1_analytic = stage1_unitary_validation()
    print(
        "Stage 1 (unitary):  "
        f"max |P1_numeric - sin^2(theta/2)| = "
        f"{np.max(np.abs(p1_numeric - p1_analytic)):.2e}"
    )

    p1_numeric, p1_analytic = stage2_T1_validation(
        T1=50.0, tlist=np.linspace(0.0, 200.0, 100)
    )
    print(
        "Stage 2 (T1):       "
        f"max |P1_numeric - exp(-t/T1)| = "
        f"{np.max(np.abs(p1_numeric - p1_analytic)):.2e}"
    )

    x_numeric, z_numeric, x_analytic = stage3_dephasing_validation(
        Tphi=40.0, tlist=np.linspace(0.0, 160.0, 100)
    )
    print(
        "Stage 3 (dephasing): "
        f"max |x_numeric - exp(-t/Tphi)| = "
        f"{np.max(np.abs(x_numeric - x_analytic)):.2e}"
    )
    print(
        "                     "
        f"max |z_numeric| = {np.max(np.abs(z_numeric)):.2e}"
    )

    benchmark = stage4_combined_benchmark(
        T1=50.0,
        T2=30.0,
        omega0=2 * np.pi * 0.05,
        tlist=np.linspace(0.0, 60.0, 200),
    )
    numeric = benchmark["numeric"]
    analytic = benchmark["analytic"]
    err4 = max(
        np.max(np.abs(np.asarray(num) - np.asarray(ref)))
        for num, ref in zip(numeric, analytic)
    )
    print(f"Stage 4 (combined): max |numeric - analytic Bloch| = {err4:.2e}")

    print("\n" + "=" * 72)
    print("Gate-fidelity worked example")
    print("=" * 72)

    T1_val, T2_val, tg_val = 50.0, 30.0, 0.04
    tlist, fidelity_curve = gate_trajectory(
        T1_val, T2_val, tg_val, theta_target=np.pi / 2
    )
    del tlist

    fast_fidelity = final_average_fidelity(
        T1_val, T2_val, tg_val, theta_target=np.pi / 2
    )
    print(
        f"T1={T1_val} us, T2={T2_val} us, tg={tg_val} us, "
        "target=Rx(pi/2)"
    )
    print(
        "Final average fidelity (trajectory): "
        f"{fidelity_curve[-1]:.6f}"
    )
    print(
        "Final average fidelity (fast function): "
        f"{fast_fidelity:.6f}"
    )

    print("\n" + "=" * 72)
    print("First-order weak-dissipation theory")
    print("=" * 72)

    for axis in ("x", "y", "z"):
        for theta in (np.pi / 4, np.pi / 2, np.pi, 3 * np.pi / 2):
            result = analytic_ab_theory(
                theta_target=theta,
                axis=axis,
            )
            print(
                f"axis={axis}, theta={theta:.4f}: "
                f"a={result['a_theory']:.8f}, "
                f"b={result['b_theory']:.8f}"
            )

    print("\n" + "=" * 72)
    print("Independent bivariate (a, b) fits")
    print("=" * 72)

    eps_grid = np.linspace(0.0005, 0.005, 6)
    for theta in (np.pi / 4, np.pi / 2, np.pi, 3 * np.pi / 2):
        fit = bivariate_ab_fit(
            eps_grid,
            eps_grid,
            theta_target=theta,
        )
        print(
            f"theta={theta:.4f}: "
            f"a={fit['a']:.6f} +/- {fit['a_se']:.1e}, "
            f"b={fit['b']:.6f} +/- {fit['b_se']:.1e}, "
            f"R^2={fit['r2']:.8f}, n={fit['n_points']}"
        )

    print("\n" + "=" * 72)
    print("Axis-independence scan")
    print("=" * 72)

    axis_results = stage10_axis_independence_scan(
        eps_max=0.005,
        n_per_axis=6,
    )
    for (axis, theta), fit in axis_results.items():
        print(
            f"axis={axis}, theta={theta:.4f}: "
            f"a={fit['a']:.6f} +/- {fit['a_se']:.1e}, "
            f"b={fit['b']:.6f} +/- {fit['b_se']:.1e}, "
            f"R^2={fit['r2']:.8f}"
        )

    print("\n" + "=" * 72)
    print("Second-order worked example")
    print("=" * 72)

    eps_grid_wide = np.linspace(0.0008, 0.03, 10)
    quadratic_fit = quadratic_ab_fit(eps_grid_wide, eps_grid_wide)

    print(
        f"a={quadratic_fit['a']:.6f}, "
        f"b={quadratic_fit['b']:.6f}, "
        f"c11={quadratic_fit['c11']:.4f}, "
        f"c22={quadratic_fit['c22']:.4f}, "
        f"c12={quadratic_fit['c12']:.4f}, "
        f"R^2={quadratic_fit['r2']:.8f}"
    )

    eps1_example, eps2_example = 8.0e-4, 1.33e-3
    direct = 1.0 - final_average_fidelity(50.0, 30.0, 0.04)
    theory = weak_dissipation_prediction(
        eps1_example,
        eps2_example,
        theta_target=np.pi / 2,
        axis="x",
    )
    print(f"Pure theory prediction = {theory:.6e}")
    print(f"Direct simulation       = {direct:.6e}")
    print(
        f"Relative difference    = "
        f"{abs(theory - direct) / direct * 100:.3f}%"
    )

    print("\n" + "=" * 72)
    print("Solver-tolerance convergence")
    print("=" * 72)

    convergence, spread = solver_convergence_check()
    for tolerance, fidelity_value in convergence.items():
        print(
            f"atol=rtol={tolerance:.0e}: "
            f"Fbar={fidelity_value:.10f}"
        )
    print(f"Maximum spread = {spread:.2e}")

    print("\n" + "=" * 72)
    print("Second-order theory at theta=pi")
    print("=" * 72)

    for axis in ("x", "z"):
        c11, c22, c12 = second_order_theory(np.pi, axis=axis)
        print(
            f"axis={axis}: c11={c11:.6f}, "
            f"c22={c22:.6f}, c12={c12:.6f}"
        )

    if generate_plots:
        print("\n" + "=" * 72)
        print("Generating original 10-plot suite")
        print("=" * 72)
        make_all_plots()

    if generate_extended_plots:
        print("\n" + "=" * 72)
        print("Generating extended plots 12-14")
        print("=" * 72)
        make_new_plots()
        make_second_order_plot()

In [ ]:
# ---------------------------------------------------------------------------
# One-cell Colab entry point
# ---------------------------------------------------------------------------
# Set these to True if you also want the figure suites generated.  The
# numerical validation/analysis is always run by default.
RUN_ORIGINAL_PLOTS = True
RUN_EXTENDED_PLOTS = True


if __name__ == "__main__":
    main(
        generate_plots=RUN_ORIGINAL_PLOTS,
        generate_extended_plots=RUN_EXTENDED_PLOTS,
    )

Stage 1-4: basic numerical validation
Stage 1 (unitary):  max |P1_numeric - sin^2(theta/2)| = 6.66e-16
Stage 2 (T1):       max |P1_numeric - exp(-t/T1)| = 4.22e-07
Stage 3 (dephasing): max |x_numeric - exp(-t/Tphi)| = 7.54e-07
                     max |z_numeric| = 0.00e+00
Stage 4 (combined): max |numeric - analytic Bloch| = 7.18e-06

Gate-fidelity worked example
T1=50.0 us, T2=30.0 us, tg=0.04 us, target=Rx(pi/2)
Final average fidelity (trajectory): 0.999423
Final average fidelity (fast function): 0.999422

First-order weak-dissipation theory
axis=x, theta=0.7854: a=0.16666667, b=0.33333333
axis=x, theta=1.5708: a=0.16666667, b=0.33333333
axis=x, theta=3.1416: a=0.16666667, b=0.33333333
axis=x, theta=4.7124: a=0.16666667, b=0.33333333
axis=y, theta=0.7854: a=0.16666667, b=0.33333333
axis=y, theta=1.5708: a=0.16666667, b=0.33333333
axis=y, theta=3.1416: a=0.16666667, b=0.33333333
axis=y, theta=4.7124: a=0.16666667, b=0.33333333
axis=z, theta=0.7854: a=0.16666667, b=0.33333333
axis=z, 

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("decoherence_figures", "zip", "figures")
files.download("decoherence_figures.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>